In [1]:
import os
import json
from typing import List, Dict, Any

from google import genai
from google.genai import types

# Read API key from environment variable
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Please set the GEMINI_API_KEY environment variable before running this notebook.")

# Initialize Gemini client
client = genai.Client(api_key=API_KEY)

In [2]:
# 2. Define Google Search tool, system prompt, and configs

# Google Search tool
search_tool = types.Tool(google_search={})

# System prompt: defines the Drug Information Agent and when to use search
DRUG_AGENT_SYSTEM_PROMPT = """
You are a Drug Information Extraction Agent for healthcare professionals and data scientists.

Your tasks:
1. Focus ONLY on drug information: indications, mechanisms of action, dosage and administration, routes,
   common and serious adverse events, warnings/contraindications, major drug-drug interactions,
   and key clinical trial highlights.
2. Always prioritize reliable sources: FDA labels, EMA, official prescribing information, major guidelines,
   or reputable medical references.
3. TOOL USAGE POLICY:
   - You have access to a google_search tool.
   - ALWAYS use google_search when:
       * The user asks for very recent information (e.g., 2023 or later label changes, new approvals,
         new safety warnings, "latest" updates, "as of now").
       * The drug is unfamiliar or rarely used.
   - You MAY answer from your internal knowledge without google_search only when:
       * The question is about well-established, stable information (classic indications,
         well-known common adverse events).
   - If you are not confident, call google_search instead of guessing.
4. You are NOT giving personal medical advice and must not make treatment decisions
   for individual patients.
5. Keep answers concise, structured, and explicitly mention uncertainties when they exist.
"""

# Config for free-form natural language / conversational answers
agent_config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=DRUG_AGENT_SYSTEM_PROMPT,
)

# Config for JSON-only responses (used for structured extraction)
json_agent_config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=DRUG_AGENT_SYSTEM_PROMPT,
    response_mime_type="application/json",
)


In [4]:
# 3. Stateless single-turn Drug Information Agent

def drug_agent(question: str) -> str:
    """
    Single-turn drug information question answering without conversation state.
    Uses gemini-2.0-flash + google_search (via agent_config).
    """
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[question],
        config=agent_config,
    )
    return response.text


In [5]:
# 4. Conversational Drug Information Agent with memory

# Global conversation history
conversation_history: List[types.Content] = []

def reset_conversation() -> None:
    """Clear the global conversation history."""
    conversation_history.clear()

def chat_drug_agent(user_message: str) -> str:
    """
    Multi-turn conversational Drug Information Agent.

    - Maintains conversation_history.
    - Decides when to use google_search according to the system prompt.
    """
    # 1) Append this user turn to the history
    conversation_history.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    # 2) Call the model with the full history
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=conversation_history,
        config=agent_config,
    )

    answer = response.text

    # 3) Append the model answer to the history
    conversation_history.append(
        types.Content(role="model", parts=[types.Part(text=answer)])
    )

    return answer


In [6]:
# 5. JSON-structured drug information extraction agent 

def drug_agent_json(question: str) -> Dict[str, Any]:
    """
    JSON-structured extraction variant.

    Expected JSON fields:
      - generic_name (str)
      - brand_names (list[str])
      - indications (list[str])
      - mechanism (str)
      - dosage_and_administration (str)
      - common_adverse_events (list[str])
      - serious_adverse_events (list[str])
      - boxed_warnings (list[str])
      - references (list[str])
    """
    json_instruction = (
        "Extract detailed drug information and respond ONLY as a JSON object with the "
        "following keys: generic_name, brand_names, indications, mechanism, "
        "dosage_and_administration, common_adverse_events, serious_adverse_events, "
        "boxed_warnings, references. Do NOT include any extra text outside the JSON object."
    )

    full_query = f"{json_instruction}\n\nUser question: {question}"

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[full_query],
        config=json_agent_config,
    )

    text = response.text
    # For debugging, you can uncomment:
    # print("RAW MODEL OUTPUT:\n", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback: return raw output for manual inspection
        return {"_parse_error": "JSONDecodeError", "_raw_text": text}

In [7]:
print (drug_agent_json("Extract detailed drug information for Keytruda."))

{'_parse_error': 'JSONDecodeError', '_raw_text': '```{\n  "generic_name": "Pembrolizumab",\n  "brand_names": [\n    "Keytruda"\n  ],\n  "indications": [\n    "Melanoma",\n    "Non-Small Cell Lung Cancer (NSCLC)",\n    "Head and Neck Squamous Cell Cancer (HNSCC)",\n    "Classical Hodgkin Lymphoma (cHL)",\n    "Microsatellite Instability-High (MSI-H) or Mismatch Repair Deficient (dMMR) Cancer",\n    "Gastric Cancer",\n    "Esophageal Cancer",\n    "Cervical Cancer",\n    "Hepatocellular Carcinoma (HCC)",\n    "Merkel Cell Carcinoma (MCC)",\n    "Renal Cell Carcinoma (RCC)",\n    "Endometrial Carcinoma",\n    "Triple-Negative Breast Cancer (TNBC)",\n    "Cutaneous Squamous Cell Carcinoma (cSCC)",\n    "Biliary Tract Cancer (BTC)"\n  ],\n  "mechanism": "Pembrolizumab is a highly selective humanized monoclonal antibody that binds to the programmed cell death-1 (PD-1) receptor and blocks its interaction with PD-L1 and PD-L2. This blockade removes PD-1 pathway-mediated inhibition of the immun

In [8]:
import json
import re
from typing import Any, Dict

def _safe_parse_json(text: str) -> Dict[str, Any]:
    """
    Try to parse model output as JSON with several fallbacks:
      1) Direct json.loads(text)
      2) Strip Markdown code fences ```...``` and retry
      3) Extract the first {...} block via regex and parse that

    Raises json.JSONDecodeError if all strategies fail.
    """
    # 1) Direct attempt
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    cleaned = text.strip()

    # 2) Remove Markdown code fences (``` or ```json etc.) and retry
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        # Drop any lines that look like ``` or ```json
        lines = [ln for ln in lines if not ln.strip().startswith("```")]
        cleaned_no_fence = "\n".join(lines).strip()
        try:
            return json.loads(cleaned_no_fence)
        except json.JSONDecodeError:
            cleaned = cleaned_no_fence  # fall through to regex step with this

    # 3) Extract the first {...} block (handles cases with multiple JSON objects)
    match = re.search(r"\{[\s\S]*?\}", cleaned)
    if match:
        candidate = match.group(0)
        return json.loads(candidate)

    # If we get here, all attempts failed
    raise json.JSONDecodeError("Unable to parse JSON from model output", text, 0)


In [9]:
# 5. JSON-structured drug information extraction agent (with robust parsing)

def drug_agent_json(question: str) -> Dict[str, Any]:
    """
    JSON-structured extraction variant.

    Expected JSON fields:
      - generic_name (str)
      - brand_names (list[str])
      - indications (list[str])
      - mechanism (str)
      - dosage_and_administration (str)
      - common_adverse_events (list[str])
      - serious_adverse_events (list[str])
      - boxed_warnings (list[str])
      - references (list[str])

    The function is tolerant to common model formatting issues:
      - Markdown code fences (``` or ```json)
      - Extra explanatory text around the JSON
      - Multiple JSON objects concatenated (it will use the first one)
    """
    json_instruction = (
        "Extract detailed drug information and respond ONLY as a JSON object with the "
        "following keys: generic_name, brand_names, indications, mechanism, "
        "dosage_and_administration, common_adverse_events, serious_adverse_events, "
        "boxed_warnings, references. Do NOT include any extra text outside the JSON object."
    )

    full_query = f"{json_instruction}\n\nUser question: {question}"

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[full_query],
        config=json_agent_config,
    )

    text = response.text
    # For debugging, you can uncomment:
    # print("RAW MODEL OUTPUT:\n", text)

    try:
        return _safe_parse_json(text)
    except json.JSONDecodeError as e:
        # Fallback: expose raw text so you can inspect what the model produced
        return {
            "_parse_error": "JSONDecodeError",
            "_error_msg": str(e),
            "_raw_text": text,
        }


In [10]:
# 6. Quick tests

print("=== Stateless drug_agent test ===")
print(drug_agent("What is Keytruda used for? List main indications and common adverse events."))

print("\n=== Conversational chat_drug_agent test ===")
reset_conversation()
print("Agent:", chat_drug_agent("Hi, I mainly treat lung cancer patients. Remember this."))
print("Agent:", chat_drug_agent("Based on my practice, what Keytruda indications are most relevant?"))
print("Agent:", chat_drug_agent("What immune-related adverse events should I watch for with Keytruda?"))

print("\n=== JSON drug_agent_json test ===")
info = drug_agent_json("Extract detailed drug information for Keytruda.")
info


=== Stateless drug_agent test ===
Keytruda (pembrolizumab) is a programmed death receptor-1 (PD-1) blocking antibody used to treat multiple cancers.

**Main Indications:**
*   Melanoma: For patients with unresectable or metastatic melanoma. Also indicated as adjuvant treatment for adult and pediatric patients (12 years and older).
*   Non-Small Cell Lung Cancer (NSCLC):

    *   First-line treatment of metastatic NSCLC in patients whose tumors have high PD-L1 expression (TPS ≥50%) with no EGFR or ALK genomic tumor aberrations.
    *   Metastatic NSCLC in patients whose tumors express PD-L1 (TPS ≥1%) with disease progression on or after platinum-containing chemotherapy. Patients with EGFR or ALK genomic tumor aberrations should have disease progression on FDA-approved therapy for these aberrations prior to receiving Keytruda.
    *   In combination with platinum-containing chemotherapy as neoadjuvant treatment, and then continued as a single agent as adjuvant treatment after surgery for

{'generic_name': 'Pembrolizumab',
 'brand_names': ['Keytruda'],
 'indications': ['Melanoma',
  'Non-Small Cell Lung Cancer (NSCLC)',
  'Head and Neck Squamous Cell Cancer (HNSCC)',
  'Classical Hodgkin Lymphoma (cHL)',
  'Microsatellite Instability-High (MSI-H) or Mismatch Repair Deficient (dMMR) Cancer',
  'Gastric Cancer',
  'Esophageal Cancer',
  'Cervical Cancer',
  'Hepatocellular Carcinoma (HCC)',
  'Merkel Cell Carcinoma (MCC)',
  'Renal Cell Carcinoma (RCC)',
  'Endometrial Carcinoma',
  'Triple-Negative Breast Cancer (TNBC)'],
 'mechanism': 'Pembrolizumab is a monoclonal antibody that binds to the programmed death receptor-1 (PD-1) protein and blocks its interaction with PD-L1 and PD-L2. This blockade removes inhibition of T-cells, leading to enhanced T-cell mediated anti-tumor immunity.',
 'dosage_and_administration': ['The dosage and administration of Keytruda varies depending on the specific indication, body weight, and whether it is used as a monotherapy or in combination 

In [11]:
# 7. Simple interactive loop inside the notebook

print("Medical Agent ready. Type 'exit' to quit this cell.")
reset_conversation()

while True:
    q = input("\nYou: ").strip()
    if q.lower() in ("exit", "quit"):
        print("Exiting chat loop.")
        break

    a = chat_drug_agent(q)
    print("Agent:", a)

Medical Agent ready. Type 'exit' to quit this cell.
Exiting chat loop.


In [12]:
# 7A. High-level dispatcher: agent_ask

def agent_ask(q: str) -> str:
    """
    High-level wrapper function used by the main interaction loop.

    Supported modes:
      - Default question: calls chat_drug_agent (multi-turn conversational mode).
      - '/json ...' command: calls drug_agent_json (structured JSON extraction).

    Example:
      /json Extract detailed drug information for Keytruda.
    """
    text = q.strip()

    # JSON extraction mode
    if text.lower().startswith("/json "):
        question = text[6:].strip()
        info = drug_agent_json(question)
        return json.dumps(info, indent=2, ensure_ascii=False)

    # Default conversational mode
    return chat_drug_agent(text)


print("Medical Agent J ready. Type 'exit' to quit.")
reset_conversation()

while True:
    q = input("\nYou: ").strip()
    if q.lower() in ("exit", "quit"):
        break

    a = agent_ask(q)
    print("Agent:", a)


Medical Agent J ready. Type 'exit' to quit.
